In [1]:
# SIGNAL PROCESSING Feature engineering
# Right-Hemisphere M1 Hand Knob
# 
# Pipeline Focus:
# 1. IIR High-pass (1 Hz) & Low-pass (100 Hz)
# 2. FIR Notch Filter (60 Hz line noise removal)
# 3. Z-score Normalization (Raw Signal)
# 4. Spectrogram Generation (1-100 Hz)
# 5. Z-Score Normalization (Per Frequency Bin in Time Domain)
# 6. Convert to Pandas DataFrame (Rows=Time, Cols=Freq)

In [2]:

# Imports
import numpy as np
import pandas as pd
from scipy import signal
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')

# Configuration
DATA_DIR  = Path("./extracted_raw_time_series")
OUTPUT_DIR = Path("./processed_data")
OUTPUT_DIR.mkdir(exist_ok=True)

FS = 500.0   # Sampling rate (Hz)


In [3]:
# Load data
print("Loading raw data...")
rest_raw = np.load(DATA_DIR / "R_M1_hand_knob_voxel_EyesClosed_Rest.npy")
move_raw = np.load(DATA_DIR / "R_M1_hand_knob_voxel_LeftHand_Move.npy")

print(f"Rest: {len(rest_raw)} samples ({len(rest_raw)/FS:.1f} s)")
print(f"Move: {len(move_raw)} samples ({len(move_raw)/FS:.1f} s)")

Loading raw data...
Rest: 7500 samples (15.0 s)
Move: 7500 samples (15.0 s)


In [4]:
# STEP 1: IIR FILTERING (High-Pass & Low-Pass)
# Applying separate High-Pass (1 Hz) and Low-Pass (100 Hz) Butterworth filters.
# Using `sosfiltfilt` for zero-phase filtering.

def apply_iir_bandpass(signal_data, fs, low_cut=1.0, high_cut=100.0, order=4):
    nyq = 0.5 * fs
    
    # 1. High-Pass Filter (remove DC drift and slow fluctuations)
    hp_wn = low_cut / nyq
    hp_sos = signal.butter(order, hp_wn, btype='highpass', output='sos')
    filtered_hp = signal.sosfiltfilt(hp_sos, signal_data)
    
    # 2. Low-Pass Filter (remove high-frequency noise/aliasing)
    lp_wn = high_cut / nyq
    lp_sos = signal.butter(order, lp_wn, btype='lowpass', output='sos')
    filtered_lp = signal.sosfiltfilt(lp_sos, filtered_hp)
    
    return filtered_lp

# Apply to both conditions
rest_filt = apply_iir_bandpass(rest_raw, FS)
move_filt = apply_iir_bandpass(move_raw, FS)

print("[IIR Filtering] Applied HP (1 Hz) and LP (100 Hz)")

[IIR Filtering] Applied HP (1 Hz) and LP (100 Hz)


In [5]:
# STEP 2: FIR NOTCH FILTER
# Removing specific line noise (60 Hz) using a FIR filter designed with `firwin`.

def apply_fir_notch(signal_data, fs, notch_freq=60.0, width=1.0, numtaps=101):
    """
    Applies a FIR band-stop (notch) filter.
    """
    nyq = 0.5 * fs
    # Define the stop band edges
    low = (notch_freq - width/2) / nyq
    high = (notch_freq + width/2) / nyq
    
    # Ensure frequencies are within valid range (0, 1)
    if low <= 0: low = 0.001
    if high >= 1: high = 0.999
    
    # Design FIR filter (Band-stop)
    taps = signal.firwin(numtaps, [low, high], pass_zero=True, fs=fs)
    
    # Apply filter (zero-phase)
    filtered_notch = signal.filtfilt(taps, 1.0, signal_data)
    return filtered_notch

# Apply Notch Filter
rest_final = apply_fir_notch(rest_filt, FS, notch_freq=60.0, width=1.0)
move_final = apply_fir_notch(move_filt, FS, notch_freq=60.0, width=1.0)

print("[FIR Notch] Applied 60 Hz Notch Filter")



[FIR Notch] Applied 60 Hz Notch Filter


In [6]:
# STEP 3: Z-SCORE NORMALIZATION (Raw Signal)
# Standardizing the signal to mean=0, std=1 before spectral analysis.

def zscore(sig_in):
    mu = np.mean(sig_in)
    sigma = np.std(sig_in)
    if sigma < 1e-12:
        return sig_in
    return (sig_in - mu) / sigma

rest_z = zscore(rest_final)
move_z = zscore(move_final)

print("[Z-Score] Raw signal normalization complete.")

[Z-Score] Raw signal normalization complete.


In [7]:
# STEP 4: SPECTROGRAM TO DATAFRAME (Time x Freq) & Z-SCORE PER FREQUENCY BIN


def create_zscored_spectrogram_df(sig, fs, nperseg=256, noverlap=None, f_min=1.0, f_max=100.0):
    """
    Computes spectrogram and returns a Pandas DataFrame:
    - Rows: Time steps
    - Columns: Frequency bins (filtered to f_min - f_max)
    - Values: Z-scored power (normalized per frequency bin across time)
    """
    if noverlap is None:
        noverlap = nperseg // 2
        
    # 1. Compute Spectrogram
    # f: frequencies, t: time segments, Sxx: power spectral density (Freq x Time)
    f, t, Sxx = signal.spectrogram(sig, fs, nperseg=nperseg, noverlap=noverlap, window='hann')
    
    # 2. Filter Frequencies (1 Hz to 100 Hz)
    freq_mask = (f >= f_min) & (f <= f_max)
    f_filtered = f[freq_mask]
    Sxx_filtered = Sxx[freq_mask, :]
    
    # 3. Convert to DataFrame
    # Initial shape: Rows=Frequencies, Columns=Time
    df_raw = pd.DataFrame(Sxx_filtered, index=f_filtered, columns=t)
    
    # 4. Transpose so Rows=Time, Columns=Frequency
    df_transposed = df_raw.T
    
    # 5. Z-Score Normalization PER FREQUENCY BIN (Column-wise)
    # We calculate mean/std for each column (frequency) across all rows (time)
    mean_vals = df_transposed.mean(axis=0)
    std_vals = df_transposed.std(axis=0)
    
    # Prevent division by zero for flat frequency bins
    std_vals[std_vals < 1e-12] = 1.0
    
    # Apply Z-score: (Value - Mean_of_Freq) / Std_of_Freq
    df_zscored = df_transposed.sub(mean_vals, axis=1).div(std_vals, axis=1)
    
    # Rename index/columns for clarity
    df_zscored.index.name = 'Time_Step'
    df_zscored.columns.name = 'Frequency_Hz'
    
    return f_filtered, t, df_zscored

# --- Configuration for Spectrogram Resolution ---
# Using 1-second windows (500 samples) for good frequency resolution
nperseg_spec = int(FS * 1.0) 
noverlap_spec = int(nperseg_spec * 0.75) # 75% overlap

print(f"Spectrogram Config: nperseg={nperseg_spec}, noverlap={noverlap_spec}")

# --- Process Rest Condition ---
print("\nComputing Z-Scored Spectrogram DataFrame for Rest...")
f_rest, t_rest, df_rest_spec_z = create_zscored_spectrogram_df(
    rest_z, FS, nperseg=nperseg_spec, noverlap=noverlap_spec, f_min=1.0, f_max=100.0
)

# --- Process Move Condition ---
print("Computing Z-Scored Spectrogram DataFrame for Move...")
f_move, t_move, df_move_spec_z = create_zscored_spectrogram_df(
    move_z, FS, nperseg=nperseg_spec, noverlap=noverlap_spec, f_min=1.0, f_max=100.0
)


Spectrogram Config: nperseg=500, noverlap=375

Computing Z-Scored Spectrogram DataFrame for Rest...
Computing Z-Scored Spectrogram DataFrame for Move...


In [8]:
df_rest_spec_z.head()

Frequency_Hz,1.0,2.0,3.0,4.0,5.0,6.0,7.0,8.0,9.0,10.0,...,91.0,92.0,93.0,94.0,95.0,96.0,97.0,98.0,99.0,100.0
Time_Step,,,,,,,,,,,,,,,,,,,,,
0.50,-0.879821,-0.590793,-0.558307,0.208577,-0.538923,-0.781652,0.802492,0.566948,-0.483574,-0.492877,...,4.575625,4.570037,2.220177,-0.643486,-1.114812,-0.671898,-1.068572,-0.051314,0.469013,-0.173966
0.75,-0.316449,-0.225447,-0.069441,-0.476802,-0.663781,-0.496803,0.616680,1.316095,0.109369,-0.526334,...,1.468372,-0.144051,-0.648512,-0.682686,-0.997936,-0.731417,0.736083,0.274432,-0.714558,0.699457
1.00,0.505066,0.431773,-0.819576,-0.228474,-0.777801,-0.400440,-1.008993,0.621760,0.045462,-0.739360,...,-0.951988,-0.280864,-1.036577,-0.899283,-0.851602,-0.958380,0.906088,-0.572977,0.499649,2.645065
1.25,0.906722,0.871879,-0.205006,-0.295015,-0.394197,-0.322259,-0.122911,-0.714701,1.234926,1.632423,...,-0.858915,-0.221022,0.589788,0.124075,-0.929396,-0.835835,-0.026280,-0.579125,1.336662,3.572486
1.50,0.385932,0.780433,-0.643466,-0.267117,-0.540970,-0.698626,0.351906,-0.415190,0.622242,2.610555,...,-0.764798,-0.470770,-0.812408,0.612199,-0.585982,-1.049267,-0.916755,-0.836168,0.728511,1.536398


In [9]:
df_move_spec_z.head()

Frequency_Hz,1.0,2.0,3.0,4.0,5.0,6.0,7.0,8.0,9.0,10.0,...,91.0,92.0,93.0,94.0,95.0,96.0,97.0,98.0,99.0,100.0
Time_Step,,,,,,,,,,,,,,,,,,,,,
0.50,-0.790607,-0.894764,-0.901118,-0.658571,-0.221159,-0.620265,0.072044,-0.625322,1.325969,4.412602,...,-0.346853,2.371333,2.329845,2.423316,0.784475,-0.168565,0.700875,-1.145137,1.958937,0.948248
0.75,-1.055199,-0.952458,-0.798199,-0.495743,1.250190,0.944459,-0.501127,-0.725687,1.521273,0.507574,...,-0.249464,-0.367993,2.156306,1.840220,-0.894984,-0.300666,0.166855,0.803006,0.265558,-0.687716
1.00,-0.516843,-0.024214,-0.101329,0.891462,0.130418,0.145593,-0.539721,-0.509824,0.845510,0.901183,...,-0.657666,-0.587725,1.400090,-0.773811,0.019657,-0.180826,-1.351517,-0.509246,-1.000433,-1.125759
1.25,-0.261247,0.902823,1.267638,2.228237,-0.538396,0.059903,-0.833391,0.223483,-1.039939,-0.214849,...,2.028415,2.004054,1.678680,1.968851,1.378513,-0.316069,-0.720193,-0.366823,-0.580931,0.758336
1.50,-0.703823,-0.128677,-0.475045,-0.931364,1.737520,1.872294,1.801277,0.838566,-0.197788,-0.711834,...,1.823328,0.172599,-0.244446,-1.005029,1.043914,-1.055632,-0.999253,-1.098476,-0.211992,-0.703210


In [ ]:
# Save to CSV
save_rest = False
save_move = False


df_rest_spec_z.to_csv(OUTPUT_DIR / "Rest_Spectrogram_ZScored_1_100Hz.csv")
print(f"Saved Rest DataFrame to {OUTPUT_DIR / 'Rest_Spectrogram_ZScored_1_100Hz.csv'}")


df_move_spec_z.to_csv(OUTPUT_DIR / "Move_Spectrogram_ZScored_1_100Hz.csv")
print(f"Saved Move DataFrame to {OUTPUT_DIR / 'Move_Spectrogram_ZScored_1_100Hz.csv'}")

